# Update Abstimmungsresultate
Datum: 28.11.2021

Immer nach den Abstimmungen werden die Abstimmungsresultate vom Datamanagement aufbereitet und auf der «[**Abstimmungs-Datenbank
von Statistik Stadt Zürich**](http://sszsttprd/statisticstools/Modules/Abstimmungen/Intranet/)» verfügbar gemacht.

Für OGD kann dann via Exportfunktion (Format Roh > Export) ein Excelfile exportiert und unter folgendem Pfad abgelegt werden: 
`\\szh.loc\ssz\applikationen\OGD\Daten\Quelldaten\SSZ\Politik\abstimmungen\__Modules_Abstimmungen_Export_AbstimmungenExport.xls`

Damit das die Ressourcen des Datasets «[Abstimmungsresultate der Stadt Zürich, seit 1933](https://data.stadt-zuerich.ch/dataset/politik-abstimmungen-seit-1933)» **manuell** aktualisiert werden können, muss das exportierte Excel noch 
 - als **Csv** aufbereitet und
 - das Attribut 'Abstimmungs_Datum' noch ins korrekte Datumsformat (**JJJJ-MM-TT / ISO 8601**) umgewandelt
 werden.
 
Früher wurden diese simplen Schritte im Libre Office (Open Office) gemacht. Nun sollen sie noch einfacher per Python gemacht werden.



<!---  
Dieses Notebook verwendet als Vorlage die Webseite «[Kitas in Berlin - Jupyter Notebook](https://juanitorduz.github.io/kitas_berlin/)» 
---!>


In [39]:
#%pip install geopandas fiona requests folium mplleaflet contextily

In [40]:
import numpy as np
import pandas as pd
import pandas as pd
import pivottablejs
from pivottablejs import pivot_ui


## Zuletzt veröffentlichte Abstimmungsresultate
Wie sehen die aktuell auf dem OGD-Katalog veröffentlichten Daten dazu aus?

In [41]:
url_abstimmungsresultate =r"https://data.stadt-zuerich.ch/dataset/politik-abstimmungen-seit-1933/download/abstimmungen_seit1933.csv"

# Read the data
bisherige_abstimmungsresultate = pd.read_csv(
    url_abstimmungsresultate, 
    sep=','    
    )
bisherige_abstimmungsresultate.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9850 entries, 0 to 9849
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Abstimmungs_Datum      9850 non-null   object 
 1   Nr_Politische_Ebene    9850 non-null   int64  
 2   Name_Politische_Ebene  9850 non-null   object 
 3   Abstimmungs_Text       9850 non-null   object 
 4   Nr_Resultat_Gebiet     9850 non-null   int64  
 5   Name_Resultat_Gebiet   9850 non-null   object 
 6   Nr_Wahlkreis_StZH      5255 non-null   float64
 7   Name_Wahlkreis_StZH    5255 non-null   object 
 8   Stimmberechtigt        6710 non-null   float64
 9   Ja                     9850 non-null   int64  
 10  Nein                   9850 non-null   int64  
 11  Stimmbeteiligung (%)   9850 non-null   float64
 12  Ja (%)                 9850 non-null   float64
 13  Nein (%)               9850 non-null   float64
 14  StaendeJa              388 non-null    object 
 15  Stae

In [42]:
bisherige_abstimmungsresultate.head(2)

,Abstimmungs_Datum,Nr_Politische_Ebene,Name_Politische_Ebene,Abstimmungs_Text,Nr_Resultat_Gebiet,Name_Resultat_Gebiet,Nr_Wahlkreis_StZH,Name_Wahlkreis_StZH,Stimmberechtigt,Ja,Nein,Stimmbeteiligung (%),Ja (%),Nein (%),StaendeJa,StaendeNein
0,2022-02-13,1,Eidgenossenschaft,Volksinitiative «Ja zum Tier- und Menschenvers...,1,Eidgenossenschaft,NaN,NaN,NaN,500937,1893539,44.2,20.9,79.1,0 0/2,20 6/2
1,2022-02-13,1,Eidgenossenschaft,Volksinitiative «Ja zum Tier- und Menschenvers...,2,Kanton Zürich,NaN,NaN,958009.0,90988,356755,47.9,20.3,79.7,NaN,NaN


## Importiere die neusten Abstimmungsresultate zur Datenaufbereitung

Wir importieren das von der Abstimmungsdatenbank exportierte Excelfile und bereiten es für die Veröffentlichung für OGD auf.


Link zu `read_excel()` in Pandas: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_excel.html

`pandas.read_excel(io, sheet_name=0, header=0, names=None, index_col=None, usecols=None, squeeze=False, dtype=None, engine=None, converters=None, true_values=None, false_values=None, skiprows=None, nrows=None, na_values=None, keep_default_na=True, na_filter=True, verbose=False, parse_dates=False, date_parser=None, thousands=None, comment=None, skipfooter=0, convert_float=None, mangle_dupe_cols=True, storage_options=None)`

In [43]:
# Filepath
fp = r"\\szh.loc\ssz\applikationen\OGD\Daten\Quelldaten\SSZ\Politik\abstimmungen\__Modules_Abstimmungen_Export_AbstimmungenExport.xls"

# Read the data
imp_abstimmungsresultate = pd.read_excel(
    io = fp, 
    parse_dates=True,
    dtype={'Abstimmungs_Text': str}
    )
imp_abstimmungsresultate.head()


,Abstimmungs_Datum,Nr_Politische_Ebene,Name_Politische_Ebene,Abstimmungs_Text,Nr_Resultat_Gebiet,Name_Resultat_Gebiet,Nr_Wahlkreis_StZH,Name_Wahlkreis_StZH,Stimmberechtigt,Ja,Nein,Stimmbeteiligung (%),Ja (%),Nein (%),StaendeJa,StaendeNein
0,2022-02-13,1,Eidgenossenschaft,Volksinitiative «Ja zum Tier- und Menschenvers...,1,Eidgenossenschaft,NaN,NaN,NaN,500937,1893539,44.2,20.9,79.1,0 0/2,20 6/2
1,2022-02-13,1,Eidgenossenschaft,Volksinitiative «Ja zum Tier- und Menschenvers...,2,Kanton Zürich,NaN,NaN,958009.0,90988,356755,47.9,20.3,79.7,NaN,NaN
2,2022-02-13,1,Eidgenossenschaft,Volksinitiative «Ja zum Tier- und Menschenvers...,3,Stadt Zürich,NaN,NaN,266675.0,25704,98575,47.9,20.7,79.3,NaN,NaN
3,2022-02-13,1,Eidgenossenschaft,Volksinitiative «Ja zum Tier- und Menschenvers...,3,Stadt Zürich,2.0,Kreis 1+2,22746.0,2446,9537,54.2,20.4,79.6,NaN,NaN
4,2022-02-13,1,Eidgenossenschaft,Volksinitiative «Ja zum Tier- und Menschenvers...,3,Stadt Zürich,3.0,Kreis 3,27831.0,2969,11343,53.1,20.7,79.3,NaN,NaN


In [44]:
imp_abstimmungsresultate.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9850 entries, 0 to 9849
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Abstimmungs_Datum      9850 non-null   datetime64[ns]
 1   Nr_Politische_Ebene    9850 non-null   int64         
 2   Name_Politische_Ebene  9850 non-null   object        
 3   Abstimmungs_Text       9850 non-null   object        
 4   Nr_Resultat_Gebiet     9850 non-null   int64         
 5   Name_Resultat_Gebiet   9850 non-null   object        
 6   Nr_Wahlkreis_StZH      5255 non-null   float64       
 7   Name_Wahlkreis_StZH    5255 non-null   object        
 8   Stimmberechtigt        6710 non-null   float64       
 9   Ja                     9850 non-null   int64         
 10  Nein                   9850 non-null   int64         
 11  Stimmbeteiligung (%)   9850 non-null   float64       
 12  Ja (%)                 9850 non-null   float64       
 13  Nei

In [54]:
latest = imp_abstimmungsresultate.query("Abstimmungs_Datum == '2022-02-13'")
latest.head(3)

,Abstimmungs_Datum,Nr_Politische_Ebene,Name_Politische_Ebene,Abstimmungs_Text,Nr_Resultat_Gebiet,Name_Resultat_Gebiet,Nr_Wahlkreis_StZH,Name_Wahlkreis_StZH,Stimmberechtigt,Ja,Nein,Stimmbeteiligung (%),Ja (%),Nein (%),StaendeJa,StaendeNein
0,2022-02-13,1,Eidgenossenschaft,Volksinitiative «Ja zum Tier- und Menschenvers...,1,Eidgenossenschaft,NaN,NaN,NaN,500937,1893539,44.2,20.9,79.1,0 0/2,20 6/2
1,2022-02-13,1,Eidgenossenschaft,Volksinitiative «Ja zum Tier- und Menschenvers...,2,Kanton Zürich,NaN,NaN,958009.0,90988,356755,47.9,20.3,79.7,NaN,NaN
2,2022-02-13,1,Eidgenossenschaft,Volksinitiative «Ja zum Tier- und Menschenvers...,3,Stadt Zürich,NaN,NaN,266675.0,25704,98575,47.9,20.7,79.3,NaN,NaN


## Gibt es beim Import Probleme?
Damit sauber exportiert werden kann, müssen die Daten auch sauber eingelesen werden. Wie können wir das aber testen?

Was sind die unique values per feature

In [46]:
imp_abstimmungsresultate.apply(lambda x: x.unique().size, axis=0)

Abstimmungs_Datum         417
Nr_Politische_Ebene         3
Name_Politische_Ebene       3
Abstimmungs_Text         2509
Nr_Resultat_Gebiet          3
Name_Resultat_Gebiet        3
Nr_Wahlkreis_StZH          17
Name_Wahlkreis_StZH        17
Stimmberechtigt          1196
Ja                       8659
Nein                     8283
Stimmbeteiligung (%)     1582
Ja (%)                    851
Nein (%)                  851
StaendeJa                  97
StaendeNein                90
dtype: int64

## Speichere die neuen Resultate in ein OGD-konformes CSV-File

OGD-konform bedeutet:
- **Datenformat**: `CSV`, 
- **Encoding**: in `UTF8`, 
- **Strings**: Zwischen Anführungszeichen `""` 
- **Separator**: ` , `

Link zu `to_csv()` in Pandas: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html

DataFrame.to_csv(path_or_buf=None, sep=',', na_rep='', float_format=None, columns=None, header=True, index=True, index_label=None, mode='w', encoding=None, compression='infer', quoting=None, quotechar='"', line_terminator=None, chunksize=None, date_format=None, doublequote=True, escapechar=None, decimal='.', errors='strict', storage_options=None)

In [53]:
export_path=r"\\szh.loc\ssz\applikationen\OGD\Daten\Quelldaten\SSZ\Politik\abstimmungen\abstimmungen_seit1933.csv"

imp_abstimmungsresultate.to_csv(
    export_path,
    encoding="UTF-8-sig",
    quotechar='"',
    quoting=2,
    index=False,
    date_format='%Y-%m-%d'
)
print('done')

# encoding="UTF-8",
#date_format='%Y-%m-%dT%H:%M:%SZ'


done


## Testing

Teste den import der eben exportierten Daten. Klappt alles wie gewollt?

In [48]:
# Read the data
checkNewCsv = pd.read_csv(
    export_path, 
    parse_dates=True,
    )

In [49]:
checkNewCsv.info(2)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9850 entries, 0 to 9849
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Abstimmungs_Datum      9850 non-null   object 
 1   Nr_Politische_Ebene    9850 non-null   int64  
 2   Name_Politische_Ebene  9850 non-null   object 
 3   Abstimmungs_Text       9850 non-null   object 
 4   Nr_Resultat_Gebiet     9850 non-null   int64  
 5   Name_Resultat_Gebiet   9850 non-null   object 
 6   Nr_Wahlkreis_StZH      5255 non-null   float64
 7   Name_Wahlkreis_StZH    5255 non-null   object 
 8   Stimmberechtigt        6710 non-null   float64
 9   Ja                     9850 non-null   int64  
 10  Nein                   9850 non-null   int64  
 11  Stimmbeteiligung (%)   9850 non-null   float64
 12  Ja (%)                 9850 non-null   float64
 13  Nein (%)               9850 non-null   float64
 14  StaendeJa              388 non-null    object 
 15  Stae

In [50]:
from pivottablejs import pivot_ui

pivot_ui(checkNewCsv)

In [51]:
checkNewCsv.query("Abstimmungs_Datum == '2022-02-13' & Nr_Politische_Ebene == 1")

,Abstimmungs_Datum,Nr_Politische_Ebene,Name_Politische_Ebene,Abstimmungs_Text,Nr_Resultat_Gebiet,Name_Resultat_Gebiet,Nr_Wahlkreis_StZH,Name_Wahlkreis_StZH,Stimmberechtigt,Ja,Nein,Stimmbeteiligung (%),Ja (%),Nein (%),StaendeJa,StaendeNein
0,2022-02-13,1,Eidgenossenschaft,Volksinitiative «Ja zum Tier- und Menschenvers...,1,Eidgenossenschaft,NaN,NaN,NaN,500937,1893539,44.2,20.9,79.1,0 0/2,20 6/2
1,2022-02-13,1,Eidgenossenschaft,Volksinitiative «Ja zum Tier- und Menschenvers...,2,Kanton Zürich,NaN,NaN,958009.0,90988,356755,47.9,20.3,79.7,NaN,NaN
2,2022-02-13,1,Eidgenossenschaft,Volksinitiative «Ja zum Tier- und Menschenvers...,3,Stadt Zürich,NaN,NaN,266675.0,25704,98575,47.9,20.7,79.3,NaN,NaN
3,2022-02-13,1,Eidgenossenschaft,Volksinitiative «Ja zum Tier- und Menschenvers...,3,Stadt Zürich,2.0,Kreis 1+2,22746.0,2446,9537,54.2,20.4,79.6,NaN,NaN
4,2022-02-13,1,Eidgenossenschaft,Volksinitiative «Ja zum Tier- und Menschenvers...,3,Stadt Zürich,3.0,Kreis 3,27831.0,2969,11343,53.1,20.7,79.3,NaN,NaN
5,2022-02-13,1,Eidgenossenschaft,Volksinitiative «Ja zum Tier- und Menschenvers...,3,Stadt Zürich,4.0,Kreis 4+5,24460.0,2400,9433,49.9,20.3,79.7,NaN,NaN
6,2022-02-13,1,Eidgenossenschaft,Volksinitiative «Ja zum Tier- und Menschenvers...,3,Stadt Zürich,5.0,Kreis 6,19564.0,2125,9196,59.6,18.8,81.2,NaN,NaN
7,2022-02-13,1,Eidgenossenschaft,Volksinitiative «Ja zum Tier- und Menschenvers...,3,Stadt Zürich,6.0,Kreis 7+8,31328.0,3630,14909,60.8,19.6,80.4,NaN,NaN
8,2022-02-13,1,Eidgenossenschaft,Volksinitiative «Ja zum Tier- und Menschenvers...,3,Stadt Zürich,7.0,Kreis 9,30971.0,3041,11118,47.1,21.5,78.5,NaN,NaN
9,2022-02-13,1,Eidgenossenschaft,Volksinitiative «Ja zum Tier- und Menschenvers...,3,Stadt Zürich,8.0,Kreis 10,24233.0,2714,10843,57.6,20.0,80.0,NaN,NaN
